# Linear Independence, Rank, and PCA Demo (Provider Features)

This notebook demonstrates:
- How to check **linear independence** of features using **matrix rank**
- How to detect **non-trivial solutions** (dependencies) via the **nullspace**
- How this connects to **dimensionality reduction (PCA)** in ML


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

# Utility to compute nullspace
def nullspace(A, rtol=1e-6):
    u, s, vh = np.linalg.svd(A)
    rankA = np.sum(s > rtol * max(A.shape) * (s[0] if len(s)>0 else 0))
    null_space = vh.T[:, rankA:]
    return null_space, s, rankA


In [ ]:
# Sample provider feature dataset
data = {
    "years_experience": [5, 10, 3, 8, 2, 15],
    "num_specialties":  [2, 3, 1, 2, 1, 4],
    "avg_rating":       [4.5, 4.7, 4.0, 4.2, 3.8, 4.9],
    # exact linear relation: total_services = 3 * num_specialties
    "total_services":   [3*n for n in [2,3,1,2,1,4]],
    "availability_score":[0.9, 0.7, 0.95, 0.8, 0.6, 0.75]
}
df = pd.DataFrame(data, index=[f"Provider_{i+1}" for i in range(len(data["years_experience"]))])
df


In [ ]:
X = df.values.astype(float)
n_features = X.shape[1]
rank = np.linalg.matrix_rank(X)

print("Matrix shape (samples, features):", X.shape)
print("Number of features:", n_features)
print("Rank of the feature matrix:", rank)
print("Are columns linearly independent?", "Yes" if rank == n_features else "No (dependent)")

# Nullspace check (non-trivial relations)
ns, svals, rankA = nullspace(X)
if ns.size > 0:
    print("\nNon-trivial relation(s) among columns:")
    for i in range(ns.shape[1]):
        vec = ns[:, i]
        if np.max(np.abs(vec)) > 0:
            readable = (vec / np.max(np.abs(vec))).round(6)
        else:
            readable = vec.round(6)
        print("relation vector (scaled):", readable)
        terms = []
        for coef, name in zip(vec, df.columns):
            coef_rounded = float(np.round(coef,6))
            if abs(coef_rounded) > 1e-9:
                terms.append(f"({coef_rounded:.6f})*{name}")
        print(" -> equation: " + " + ".join(terms) + " = 0\n")


In [ ]:
pca = PCA()
pca.fit(X)
explained = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained)

print("Explained variance ratio:", np.round(explained, 4))
print("Cumulative explained variance:", np.round(cum_explained, 4))

k = rank
pca_k = PCA(n_components=k)
X_reduced = pca_k.fit_transform(X)
pd.DataFrame(X_reduced, index=df.index, columns=[f"PC{i+1}" for i in range(k)])


In [1]:
import numpy as np

# ------------------------------
# 1. Define some sample vectors
# ------------------------------
v1 = np.array([1, 0, 0])
v2 = np.array([0, 1, 0])
v3 = np.array([1, 1, 0])
v4 = np.array([0, 0, 1])

# Put them as columns of a matrix
A = np.column_stack([v1, v2, v3, v4])
print("Matrix A (columns are vectors):\n", A)

# ------------------------------
# 2. Check Linear Independence
# ------------------------------
rank = np.linalg.matrix_rank(A)
n_vectors = A.shape[1]

print("\nNumber of vectors:", n_vectors)
print("Rank of matrix (max # independent vectors):", rank)

if rank == n_vectors:
    print("✅ All vectors are linearly independent")
else:
    print("❌ Vectors are linearly dependent")

# ------------------------------
# 3. Trivial vs Non-Trivial Solutions
# Solve A·x = 0
# ------------------------------
u, s, vh = np.linalg.svd(A)
tol = 1e-9
null_mask = (s <= tol)
null_space = vh.T[:, len(s):]  # extra columns give nullspace
if null_space.size == 0:
    print("\nOnly trivial solution exists: x = 0")
else:
    print("\nNon-trivial solution exists (dependency found):")
    for vec in null_space.T:
        print("Nullspace vector:", vec.round(3))
        # Means a combination of the columns = 0

# ------------------------------
# 4. Span of vectors
# ------------------------------
# Span = all linear combinations of given vectors
print("\nSpan of {v1, v2, v3}: lies in the xy-plane (R^2 in R^3)")
print("Because v3 = v1 + v2 (so it adds no new direction).")

# ------------------------------
# 5. Basis Vectors
# ------------------------------
# A basis = minimal independent set of vectors that spans the same space
# np.linalg.qr can help us extract basis
Q, R = np.linalg.qr(A)
basis = Q[:, :rank]

print("\nBasis vectors for the column space:")
print(basis.round(3))


Matrix A (columns are vectors):
 [[1 0 1 0]
 [0 1 1 0]
 [0 0 0 1]]

Number of vectors: 4
Rank of matrix (max # independent vectors): 3
❌ Vectors are linearly dependent

Non-trivial solution exists (dependency found):
Nullspace vector: [ 0.577  0.577 -0.577  0.   ]

Span of {v1, v2, v3}: lies in the xy-plane (R^2 in R^3)
Because v3 = v1 + v2 (so it adds no new direction).

Basis vectors for the column space:
[[ 1.  0.  0.]
 [-0.  1.  0.]
 [-0. -0.  1.]]


In [2]:
import numpy as np
import pandas as pd

# ------------------------------
# 1. Create a provider dataset
# ------------------------------
data = {
    "years_experience": [5, 10, 3, 8, 2],
    "num_specialties":  [2, 3, 1, 2, 1],
    "avg_rating":       [4.5, 4.7, 4.0, 4.2, 3.8],
    "total_services":   [6, 9, 3, 6, 3],   # 3 × num_specialties
    "services_per_specialty": [3, 3, 3, 3, 3] # = total_services / num_specialties
}
df = pd.DataFrame(data)
print("Provider Dataset:\n", df)

# ------------------------------
# 2. Check linear dependence
# ------------------------------
X = df.values.astype(float)
rank = np.linalg.matrix_rank(X)
n_features = X.shape[1]

print("\nNumber of features:", n_features)
print("Rank of matrix:", rank)

if rank < n_features:
    print("❌ Features are linearly dependent → redundant features exist")
else:
    print("✅ Features are independent")

# ------------------------------
# 3. Find non-trivial solution (nullspace)
# ------------------------------
u, s, vh = np.linalg.svd(X)
tol = 1e-9
rankA = np.sum(s > tol)
null_space = vh.T[:, rankA:]

if null_space.size > 0:
    print("\nNon-trivial relation(s) among features:")
    for vec in null_space.T:
        print("Dependency vector:", vec.round(3))


Provider Dataset:
    years_experience  num_specialties  avg_rating  total_services  \
0                 5                2         4.5               6   
1                10                3         4.7               9   
2                 3                1         4.0               3   
3                 8                2         4.2               6   
4                 2                1         3.8               3   

   services_per_specialty  
0                       3  
1                       3  
2                       3  
3                       3  
4                       3  

Number of features: 5
Rank of matrix: 4
❌ Features are linearly dependent → redundant features exist

Non-trivial relation(s) among features:
Dependency vector: [-0.     0.949  0.    -0.316 -0.   ]
